In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
import mlflow

/home/aman/Applied-ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def score_model(model, X, y):
    y_pred = model.predict(X)
    scores = {
            'precision': precision_score(y, y_pred),
            'recall': recall_score(y, y_pred),
            'f1_score': f1_score(y, y_pred)
    }
    return scores

def evaluate_predictions(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    
    return cm, report


def create_model(model_type, params):
    if model_type == 'logistic_regression':
        classifier = LogisticRegression(**params)
    elif model_type == 'naive_bayes':
        classifier = MultinomialNB(**params)
    elif model_type == 'random_forest':
        classifier = RandomForestClassifier(**params)

    model = Pipeline([('vectorizer', TfidfVectorizer(stop_words = 'english')),
                      ('model', classifier)])
    return model

def train(experiment: dict):
    model_type = experiment['model_type']
    params = experiment["params"]

    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_experiment("spam-filter-track0")
    mlflow.autolog()

    # Load datasets
    train = pd.read_csv('../data/train.csv')
    validation = pd.read_csv('../data/validation.csv')

    # Split datasets
    X_train, y_train = train['message'], train['label']
    X_validation, y_validation = validation['message'], validation['label']

    # Create model
    model = create_model(model_type, params)

    # Train model
    with mlflow.start_run(run_name=f"{model_type}_experiment"):

        mlflow.log_param("model_type", model_type)
        mlflow.log_params(params)

        model.fit(X_train, y_train)
        y_validation_pred = model.predict(X_validation)
        y_validation_scores = model.predict_proba(X_validation)[:, 1]


        cm, report = evaluate_predictions(y_validation, y_validation_pred)
        scores = score_model(model, X_validation, y_validation)
        aucpr = average_precision_score(y_validation, y_validation_scores)
        scores["aucpr"] = aucpr

        mlflow.log_metrics(scores)
        mlflow.log_text(str(cm), "confusion_matrix.txt")
        mlflow.log_text(report, "classification_report.txt")
        model_info = mlflow.sklearn.log_model(model, "model")

        mlflow.register_model(model_uri=model_info.model_uri, name="spam-filter-classifier")    

### Logistic Regression Experiment

In [3]:
logistic_regression_exp = {
    "model_type": "logistic_regression",
    "params": {
        "max_iter": 1000
    }
}

train(logistic_regression_exp)

2026/03/08 13:58:03 INFO mlflow.tracking.fluent: Experiment with name 'spam-filter-track0' does not exist. Creating a new experiment.
2026/03/08 13:58:04 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/08 13:58:04 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 13:58:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 13:58:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/03/08 13:58:13 WARNING mlflow.sklearn: Unrecognized data

🏃 View run logistic_regression_experiment at: http://127.0.0.1:5000/#/experiments/2/runs/5680d46f6c3e4fed848a69cbd6bd3121
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### Naive Bayes Experiment

In [4]:
naive_bayes_exp = {
    "model_type": "naive_bayes",
    "params": {}
}
train(naive_bayes_exp)

2026/03/08 13:59:54 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/08 13:59:55 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 13:59:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 14:00:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/03/08 14:00:02 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 14:00:03 WARNING mlflow.sklearn: Unrecognized datase

🏃 View run naive_bayes_experiment at: http://127.0.0.1:5000/#/experiments/2/runs/fb4c90dc4f1848ed948446d74b80e319
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### Random Forest Experiment

In [5]:
random_forest_exp = {
    "model_type": "random_forest",
    "params": {
        "n_estimators": 100,
        "random_state": 42
    }
}
train(random_forest_exp)

2026/03/08 14:01:48 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/08 14:01:48 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 14:01:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 14:01:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/03/08 14:01:54 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 14:01:59 WARNING mlflow.sklearn: Unrecognized datase

🏃 View run random_forest_experiment at: http://127.0.0.1:5000/#/experiments/2/runs/b2148fa873734595842c3e944cdb4c51
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### AUCPR For Three Benchmark models

In [6]:
test = pd.read_csv("../data/test.csv")

X_test = test["message"]
y_test = test["label"]

In [9]:
model_v1 = mlflow.sklearn.load_model("models:/spam-filter-classifier/1")
model_v2 = mlflow.sklearn.load_model("models:/spam-filter-classifier/2")
model_v3 = mlflow.sklearn.load_model("models:/spam-filter-classifier/3")

In [10]:
probs_v1 = model_v1.predict_proba(X_test)[:, 1]
probs_v2 = model_v2.predict_proba(X_test)[:, 1]
probs_v3 = model_v3.predict_proba(X_test)[:, 1]

2026/03/08 14:16:13 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 14:16:13 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 14:16:13 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.


In [11]:
aucpr_v1 = average_precision_score(y_test, probs_v1)
aucpr_v2 = average_precision_score(y_test, probs_v2)
aucpr_v3 = average_precision_score(y_test, probs_v3)

print("Model 1 AUCPR:", aucpr_v1)
print("Model 2 AUCPR:", aucpr_v2)
print("Model 3 AUCPR:", aucpr_v3)

Model 1 AUCPR: 0.9576027855999234
Model 2 AUCPR: 0.9426560636258038
Model 3 AUCPR: 0.9607478653757539
